# Cropped ShallowFBCSP Development and Fusion
Exploratory whole-trial, fixed raw-crop, strict TTA, and guarded same-split fusion development.

# 1. Setup

In [ ]:
from __future__ import annotations
import builtins, hashlib, json, os, platform, random, sys, time
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from modern_mi_common import *
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | CWD: {Path.cwd()}")

# 2. Configuration
## 2.1 Locked Crop Defaults
## 2.2 CONFIG

In [ ]:
WORKING_DIR=Path.cwd().resolve().parent.parent
CONFIG = {
    # Paths / run identity
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-cropped-shallow-fusion"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "cropped_shallow_20epoch_primary", "config_note": "Exploratory development; not independent confirmation.",
    # Dataset / independent preprocessing
    "subjects_to_use": None, "channel_set": "liu29", "native_sfreq": 500, "target_sfreq": 128, "marker_channel_index": 32, "onset_marker_value": 2, "onset_plausible_range": [800,1300], "window_seconds": 4.0, "bandpass_hz": [4.0,40.0], "filter_order": 4, "normalization_mode": "channel_standardize", "normalization_eps": 1e-6,
    # Fixed crop protocol: split original trials first
    "strategy": "cropped_raw", "crop_seconds": 2.0, "crop_starts_seconds": [0.0,0.5,1.0,1.5,2.0], "dense_prediction_alternative": False,
    "cv_folds": 5, "cv_random_state": 2026, "expected_global_split_hash": "801ec1d2c981335f",
    # ShallowFBCSP / optimization
    "n_filters_time": 40, "n_filters_spat": 40, "batch_size": 8, "n_epochs": 20, "learning_rate": 0.0003, "weight_decay": 0.01, "gradient_clip_norm": 1.0,
    "seed": 2026, "set_seed": True, "bootstrap_iterations": 10000, "collapse_threshold": 0.9,
    # Fusion is fail-closed until explicitly enabled; no artifact predictions are accepted
    "enable_riemann_fusion": False, "fusion_implementation": "disabled_neural_only", "fusion_weight_neural": 0.5, "fusion_weight_riemann": 0.5
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp=datetime.now().strftime("%Y%m%d_%H%M%S_%f"); config_hash=hashlib.md5(json.dumps(CONFIG,sort_keys=True,default=str).encode()).hexdigest()[:8]; return f"{timestamp}_{config_hash}"
RUN_ID=create_run_id(); ARTIFACT_DIR=Path(CONFIG["artifact_dir"])/RUN_ID; ARTIFACT_DIR.mkdir(parents=True,exist_ok=False); LOG_PATH=ARTIFACT_DIR/"run.log"; _LOG_FILE_HANDLE=open(LOG_PATH,"a",buffering=1,encoding="utf-8",errors="replace")
def _safe_write_text(stream,text):
    try: stream.write(text)
    except UnicodeEncodeError:
        enc=getattr(stream,"encoding",None) or "utf-8"; stream.write(text.encode(enc,errors="replace").decode(enc,errors="replace"))
def _timestamped_print(*args,**kwargs):
    sep=kwargs.pop("sep"," " ); end=kwargs.pop("end","\n"); flush=kwargs.pop("flush",False); file=kwargs.pop("file",None); target=sys.stdout if file is None else file; stamped=f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {sep.join(str(a) for a in args)}"; _safe_write_text(target,stamped+end); _safe_write_text(_LOG_FILE_HANDLE,stamped+end)
    if flush: target.flush(); _LOG_FILE_HANDLE.flush()
builtins.print=_timestamped_print; config_path=ARTIFACT_DIR/"config.json"; config_path.write_text(json.dumps(CONFIG,indent=2),encoding="utf-8")
print(f"Run ID:     {RUN_ID}"); print(f"Artifacts:  {ARTIFACT_DIR}"); print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")
DEVICE=resolve_device()
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"]=str(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True,warn_only=True)
BASE_SEED=int(CONFIG["seed"]); seed_everything(BASE_SEED); print(f"Using device: {DEVICE}")

# 3. Load and Prepare Data
## 3.1 Shared Liu2024 Loading Helpers
## 3.2 Split-First Fixed Crop Pipeline
Original trial indices are split before any crops are generated. The five fixed 2 s crops start at 0, 0.5, 1, 1.5, and 2 s.
## 3.3 Trial-Balanced Crop Dataset
Every training trial contributes exactly five crops, so mean crop loss weights original trials equally.
## 3.4 Locate and Load Data

In [ ]:
def crop_sample_starts(cfg): return [int(round(s*cfg["target_sfreq"])) for s in cfg["crop_starts_seconds"]]
def fixed_crops(x,cfg,trial_indices):
    starts=crop_sample_starts(cfg); length=int(round(cfg["crop_seconds"]*cfg["target_sfreq"])); trial_indices=np.asarray(trial_indices,int)
    if any(s<0 or s+length>x.shape[-1] for s in starts): raise AssertionError("A fixed crop exceeds its original 4 s trial")
    crops=np.stack([x[t,:,s:s+length] for t in trial_indices for s in starts]); owners=np.repeat(trial_indices,len(starts)); offsets=np.tile(starts,len(trial_indices)); return crops,owners,offsets
def assert_crop_split(train_idx,test_idx,train_owners,test_owners,cfg):
    if set(train_idx)&set(test_idx): raise AssertionError("Outer train/test original trials overlap")
    if set(train_owners)-set(train_idx) or set(test_owners)-set(test_idx) or set(train_owners)&set(test_owners): raise AssertionError("Crop crossed an outer fold")
    expected=len(cfg["crop_starts_seconds"]); counts=np.bincount(train_owners,minlength=40)[train_idx]
    if not np.all(counts==expected): raise AssertionError("Crop loss is not trial-balanced")
def collapse_logits(logits):
    if logits.ndim==2: return logits
    return logits.flatten(start_dim=2).mean(-1)
def make_shallow(n_chans,n_times,dense=False):
    kwargs={"n_filters_time":CONFIG["n_filters_time"],"n_filters_spat":CONFIG["n_filters_spat"],"final_conv_length":1 if dense else "auto"}; return build_model("ShallowFBCSPNet",n_chans,n_times,CONFIG["target_sfreq"],DEVICE,kwargs)
def train_shallow(model,x_train,y_train):
    loader=DataLoader(TensorDataset(torch.from_numpy(x_train),torch.from_numpy(y_train).long()),batch_size=CONFIG["batch_size"],shuffle=True,generator=torch.Generator().manual_seed(BASE_SEED)); opt=torch.optim.AdamW(model.parameters(),lr=CONFIG["learning_rate"],weight_decay=CONFIG["weight_decay"]); curve=[]; start=time.perf_counter()
    for epoch in range(CONFIG["n_epochs"]):
        model.train(); total=0.0; count=0
        for xb,yb in loader:
            opt.zero_grad(set_to_none=True); logits=collapse_logits(model(xb.to(DEVICE))); loss=nn.functional.cross_entropy(logits,yb.to(DEVICE)); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),CONFIG["gradient_clip_norm"]); opt.step(); total+=float(loss.detach())*len(xb); count+=len(xb)
        curve.append(total/count)
    return curve,time.perf_counter()-start
def predict_trials(model,x,cfg,strategy):
    model.eval()
    if strategy=="whole": batches=x[:,None,:,:]
    else:
        crops,owners,_=fixed_crops(x,cfg,np.arange(len(x))); batches=crops.reshape(len(x),len(cfg["crop_starts_seconds"]),x.shape[1],-1)
    trial_logits=[]
    with torch.no_grad():
        for trial_views in batches:
            logits=collapse_logits(model(torch.from_numpy(trial_views.astype("float32")).to(DEVICE))); trial_logits.append(logits.mean(0).cpu().numpy())
    logits=np.stack(trial_logits); logits-=logits.max(1,keepdims=True); prob=np.exp(logits); return prob/prob.sum(1,keepdims=True)
def guard_fusion():
    if CONFIG["enable_riemann_fusion"]: raise RuntimeError("Riemannian fusion is intentionally disabled in this neural-first implementation. Enable only after same-fold motor13 OAS/tangent 1s+2s predictions and training-only cross-fitted score calibration are implemented here; artifact predictions are forbidden.")

# 4. Model
## 4.1 ShallowFBCSP Strategies
`cropped_raw` is primary. `whole` is the control. `strict_tta` trains one dense (`final_conv_length=1`) whole-trial model and averages logits over only the same fixed crop set. `dense_whole` is an explicit alternative.
## 4.2 Fusion Guard
Fusion cannot consume mismatched artifact predictions and is disabled until a same-outer-fold motor13 OAS/tangent 1 s + 2 s branch with training-only cross-fitted score calibration is implemented in this notebook.

# 5. Training
## 5.1 Fixed Optimization and Loss Curves
## 5.2 Split-First Within-Subject Runner
## 5.3 Run All Subjects

In [ ]:
print("Full CONFIG banner:\n"+json.dumps(CONFIG,indent=2)); guard_fusion(); strategy=CONFIG["strategy"]
if strategy not in {"whole","cropped_raw","strict_tta","dense_whole"}: raise ValueError(f"Unknown strategy {strategy}")
paths=locate_subject_files(CONFIG["source_extract_dir"],CONFIG["subjects_to_use"]); SUBJECTS=[subject_id(p) for p in paths]; loaded={}; all_splits={}; inventory=[]
for p in paths:
    sid=subject_id(p); x,y,CH_NAMES,onsets=load_subject(p,CONFIG); splits=make_splits(y,CONFIG,sid); loaded[sid]=(x,y); all_splits[sid]=splits; inventory.append({"subject_id":sid,"path":str(p),"n_trials":40,"subject_split_hash":stable_hash(splits)})
split_hashes={stable_hash(v) for v in all_splits.values()}
if split_hashes!={CONFIG["expected_global_split_hash"]}: raise AssertionError(f"Subject split hashes {split_hashes} != locked {CONFIG['expected_global_split_hash']}")
GLOBAL_SPLIT_HASH=CONFIG["expected_global_split_hash"]
FOLD_RESULTS=[]
for sid in SUBJECTS:
    x,y=loaded[sid]
    for split in all_splits[sid]:
        seed_everything(BASE_SEED); tr=np.asarray(split["train_indices"]); te=np.asarray(split["test_indices"]); mean,scale=fit_normalizer(x[tr],CONFIG["normalization_mode"],CONFIG["normalization_eps"]); xn=((x-mean)/scale).astype("float32"); train_crops,train_owners,_=fixed_crops(xn,CONFIG,tr); test_crops,test_owners,_=fixed_crops(xn,CONFIG,te); assert_crop_split(tr,te,train_owners,test_owners,CONFIG)
        if strategy=="cropped_raw": train_x=train_crops; train_y=np.repeat(y[tr],len(CONFIG["crop_starts_seconds"])); model=make_shallow(x.shape[1],train_x.shape[-1],False); infer_strategy="crops"
        elif strategy=="strict_tta": train_x=xn[tr]; train_y=y[tr]; model=make_shallow(x.shape[1],x.shape[-1],True); infer_strategy="crops"
        else: train_x=xn[tr]; train_y=y[tr]; model=make_shallow(x.shape[1],x.shape[-1],strategy=="dense_whole"); infer_strategy="whole"
        curve,elapsed=train_shallow(model,train_x,train_y); prob=predict_trials(model,xn[te],CONFIG,infer_strategy); pred=prob.argmax(1); FOLD_RESULTS.append(fold_result(sid,split["fold_id"],te,y[te],pred,prob,model,elapsed,BASE_SEED,{"strategy":strategy,"training_loss_curve":curve,"n_train_original_trials":len(tr),"n_train_examples":len(train_x),"crop_starts_samples":crop_sample_starts(CONFIG),"trial_level_aggregation":"mean_logits","global_split_hash":GLOBAL_SPLIT_HASH}))
    observed=sorted(i for r in FOLD_RESULTS if r["subject_id"]==sid for i in r["test_indices"]); assert observed==list(range(40)), f"{sid}: expected exactly one final prediction per original trial"
subject_inventory_path=ARTIFACT_DIR/"subject_inventory.csv"; pd.DataFrame(inventory).to_csv(subject_inventory_path,index=False); (ARTIFACT_DIR/"splits.json").write_text(json.dumps(all_splits,indent=2),encoding="utf-8")

# 6. Results
## 6.1 Original-Trial Aggregate Metrics

In [ ]:
SUBJECT_ROWS=[]
for sid in SUBJECTS:
    rr=[r for r in FOLD_RESULTS if r["subject_id"]==sid]; idx=np.concatenate([r["test_indices"] for r in rr]); order=np.argsort(idx); yt=np.concatenate([r["true_labels"] for r in rr])[order]; yp=np.concatenate([r["predictions"] for r in rr])[order]; SUBJECT_ROWS.append({"subject_id":sid,"accuracy":float(accuracy_score(yt,yp)),"balanced_accuracy":float(balanced_accuracy_score(yt,yp)),"n_original_trials":40})
vals=[r["balanced_accuracy"] for r in SUBJECT_ROWS]; GLOBAL_METRICS={"mean_subject_balanced_accuracy":float(np.mean(vals)),"subject_bootstrap_95_ci":bootstrap_ci(vals,BASE_SEED,CONFIG["bootstrap_iterations"]),"n_subjects":len(vals),"n_folds_total":len(FOLD_RESULTS),"collapse_rate":float(np.mean([r["collapse_diagnostics"]["collapsed"] for r in FOLD_RESULTS])),"global_split_hash":GLOBAL_SPLIT_HASH,"strategy":strategy,"inference_unit":"original_trial","development_status":"exploratory_not_independent_confirmation","riemann_fusion_enabled":False}

## 6.2 Performance Visualizations

In [ ]:
plot_path=ARTIFACT_DIR/"cropped_shallow_subject_performance.png"; plt.figure(figsize=(10,3)); plt.bar([r["subject_id"] for r in SUBJECT_ROWS],[100*r["balanced_accuracy"] for r in SUBJECT_ROWS]); plt.axhline(50,color="k",ls="--"); plt.xticks(rotation=90); plt.tight_layout(); plt.savefig(plot_path,dpi=150); plt.close()
loss_path=ARTIFACT_DIR/"training_loss_curves.png"; plt.figure(figsize=(6,3)); [plt.plot(r["training_loss_curve"],alpha=.15,color="tab:orange") for r in FOLD_RESULTS]; plt.xlabel("Epoch"); plt.ylabel("Training loss"); plt.tight_layout(); plt.savefig(loss_path,dpi=150); plt.close()

## 6.3 Experiment Summary
## 6.4 Crop and Fusion Diagnostics

In [ ]:
print(json.dumps(GLOBAL_METRICS,indent=2))

## 6.5 Save Artifacts

In [ ]:
cv_results_path=ARTIFACT_DIR/"cv_results.json"; cv_results_path.write_text(json.dumps(FOLD_RESULTS,indent=2),encoding="utf-8")
subject_metrics_path=ARTIFACT_DIR/"subject_metrics.json"; subject_metrics_path.write_text(json.dumps(SUBJECT_ROWS,indent=2),encoding="utf-8")
global_metrics_path=ARTIFACT_DIR/"global_metrics.json"; global_metrics_path.write_text(json.dumps(GLOBAL_METRICS,indent=2),encoding="utf-8"); pd.DataFrame(SUBJECT_ROWS).to_csv(ARTIFACT_DIR/"subject_results.csv",index=False)
run_metadata={"run_id":RUN_ID,"artifact_dir":str(ARTIFACT_DIR),"experiment_name":CONFIG["experiment_name"],"config_note":CONFIG["config_note"],"subjects":SUBJECTS,"channel_names":CH_NAMES,"model_name":"ShallowFBCSPNet","strategy":strategy,"seed":BASE_SEED,"global_split_hash":GLOBAL_SPLIT_HASH,"global_metrics":GLOBAL_METRICS,"artifacts":{p.name:str(p) for p in ARTIFACT_DIR.iterdir()}}; run_metadata_path=ARTIFACT_DIR/"run_metadata.json"; run_metadata_path.write_text(json.dumps(run_metadata,indent=2),encoding="utf-8")
print(f"CV results saved to:      {cv_results_path}"); print(f"Subject metrics saved to: {subject_metrics_path}"); print(f"Global metrics saved to:  {global_metrics_path}"); print(f"Run metadata saved to:    {run_metadata_path}"); print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass